# Non-linear effects from infectious disease models

This notebook will explore how non-linear effects of
public health interventions against infectious diseases
can be reflected in mechanistic models such as the one
introduced in the previous notebook.

We'll start off with the similar package installation code
(just a couple more commonly used Python packages),
which you can ignore.

In [ ]:
%pip install summerepi2==1.3.6
import numpy as np
import pandas as pd
from plotly import express as px
from plotly import graph_objects as go
from summer2 import CompartmentalModel
from summer2.parameters import Parameter

Next, we'll build a model that is very similar to the one in notebook 01,
except that we won't add the process of infection yet.

In [ ]:
total_population = 1e6  
infectious_seed = 1.0
run_period = [0.0, 50.0]
model_comps = ["susceptible", "infectious", "recovered"]
infect_comps = ["infectious"]
sir_model = CompartmentalModel(times=run_period, compartments=model_comps, infectious_compartments=infect_comps, timestep=0.2)
start_pop = {"susceptible": total_population - infectious_seed, "infectious": infectious_seed}
sir_model.set_initial_population(start_pop)
sir_model.add_transition_flow(name="recovery", fractional_rate=Parameter("recovery_rate"), source="infectious", dest="recovered")
base_parameters = {
    "contact_rate": 1.5,
    "recovery_rate": 0.2,
}

## Adding an intervention
Let's consider the effect of a public health intervention on the
dynamics we observed with the base model in the previous noteboook.
To capture this, we'll imagine that the government has implemented
some policies that reduce the rate at which people come into
contact with one-another, i.e. a "lockdown".
We'll implement this in the model by assuming some strength
of the effect of lockdown on transmission, so that a stronger
lockdown reduces transmission further.
The following cell achieves this in code
by scaling back the rate at which people get infected
(for a certain prevalence of infection) according to
the effect of the intervention.

In [ ]:
infection_process = Parameter("contact_rate") * (1.0 - Parameter("lockdown_effect"))
sir_model.add_infection_frequency_flow(name="infection", contact_rate=infection_process, source="susceptible", dest="infectious")

Now let's run the model with various lockdown effect strengths and see how it
plays out in the infection dynamics.

In [ ]:
lockdown_effects = np.linspace(0.0, 1.0, 10)
outputs = pd.DataFrame(columns=lockdown_effects)
for effect in lockdown_effects:
    parameters = base_parameters | {"lockdown_effect": effect}
    sir_model.run(parameters)
    outputs[effect] = sir_model.get_outputs_df()["infectious"]


First let's consider what that looks like in the epi-curve of cases over time.

In [ ]:
fig = go.Figure()
legend_format = {"title": "lockdown effect"}
xaxis_format = {"title": "days"}
for c in outputs.columns:
    colour = f"rgba({c}, 0, {1.0 - c}, 1)"
    fig.add_trace(go.Scatter(x=outputs.index, y=outputs[c], mode="lines", name=c, line={"color": colour}))
fig.update_layout(title="effect of lockdown severity on epi-curve", legend=legend_format, xaxis=xaxis_format, yaxis={"title": "infection prevalence"})

Next let's have a look at how that looks in terms of cumulative cases
rather than cases per day.

In [ ]:
# Calculate the cumulative numbers from the daily
cum_outputs = outputs.cumsum()

cum_yaxis_format = {"title": "cumulative cases"}
cum_fig = go.Figure()
for c in cum_outputs.columns:
    colour = f"rgba({c}, 0, {1.0 - c}, 1)"
    cum_fig.add_trace(go.Scatter(x=cum_outputs.index, y=cum_outputs[c], mode="lines", name=c, line={"color": colour}))
cum_fig.update_layout(title="effect of lockdown severity on cumulative cases", legend=legend_format, xaxis=xaxis_format, yaxis=cum_yaxis_format)

Rather than looking at the cumulative cases over time,
the non-linear effect might be clearer
if we look at the size of the epidemic by 50 days
for various lockdown strengths.

In [ ]:
cum_cases_day50 = cum_outputs.iloc[-1]
nonlinear_fig = go.Figure()
nonlinear_fig.add_trace(go.Scatter(x=cum_outputs.columns, y=cum_cases_day50))
nonlinear_fig.update_layout(title="cumulative cases at 50 days by lockdown effect", xaxis={"title": "lockdown effect"}, yaxis=cum_yaxis_format)